# ADS v2: shared working tables (sales

Builds the shared working tables: `policies_sold`, `relay_document_event`, `renewal_cycles`, `renewal_scv_quotes`, `hfq_priced`, `hfq_payment`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("10_Common", "started")


## Shared: policies_sold

Active sales before the effective date (notebook 04 cell 17, stg.GlobalPoliciesSold). Every journey reads it.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_policies_sold                                                               */
/*  Description : Active sales (Cat A1, OrgID 1, SaleCount 1) before the effective date.                */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicySale                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_policies_sold as
select  PolicyId, PolicyCode, Channel, ReportingSaleType, PolicyTypeGroup, QuoteQueryGuid, HEQReference, ReportingSaleDate
from    ${ads.ods}.PolicySale
where   EffectiveDate < date '${ads.effective_date}'
and     ReportingSaleCategory = 'Cat A1: Active Sale'
and     OrgID = 1
and     SaleCount = 1

## Shared: relay_document_event

Relay document events from a month before the window to a month after it (families apply their own window), as notebooks 05 / 10 built ods.EventStream source 3: the description is 'business process - event type - document' (null when the event type has no name, as before), the policy's product from Policy (Motor / Home / Van / Other), and cash receipts excluded (change of 2026-07-23).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_relay_document_event                                                        */
/*  Description : Relay document events per policy around the window, with the old EventStream descript */
/*              : ion and product.                                                                      */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyDocumentEvent, ods_v2.Policy                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_relay_document_event as
select  d.EventId,
        d.DocumentId,
        d.PolicyCode,
        d.PolicyId,
        p.PolicyTypeGroup,
        d.EventDate                                                     as EventDateTime,
        concat(d.BusinessProcess, ' - ', d.EventType, ' - ', d.DocumentDescription) as EventDescription
from    ${ads.ods}.PolicyDocumentEvent d
join    ${ads.ods}.Policy p     on p.PolicyCode = d.PolicyCode
where   p.PolicyTypeGroup in ('Motor', 'Home', 'Van', 'Other')
and     d.EventDate >= add_months(timestamp '${ads.start_time}', -1)
and     d.EventDate <  add_months(timestamp '${ads.end_time}', 1)
and     coalesce(concat(d.BusinessProcess, ' - ', d.EventType, ' - ', d.DocumentDescription), '')
            not like 'Unknown - Saved Document - Cash Receipt No: %'

## Shared: renewal_cycles

Renewal cycles for every product (D-E1.19, D-E1.21, D-E1.26): the finance team's cycles (PolicyRenewal, PolicyOfferNum = 1, renew date from the window start) with their PolicyOfferNum / PolicyRetNum, plus the operations team's +/-30-day rule on top: a cycle is also renewed when the same customer bought a policy of the same product starting within 30 days either side of the end date (the segmentation V3 matching, one-to-one, best evidence first: same code, warehouse lineage, same vehicle / address, date only). The resolved renewal (finance retained, else the +/-30-day match) gives the renewal date, policy and channel; online = renewal channel 'e) Online' or a portal payment, else call centre. Lapsed = no renewal by the due date; renewed inside the window = renewed after the due date and within due + 7 days online or due + 30 days call centre; lost = not renewed within that window. Outcomes are final once due + 30 days is before the window end.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_terms                                                               */
/*  Description : The finance renewal cycles with the customer (SCV key by client code), address and ve */
/*              : hicle keys, and the end date (cancellation inside the term, else the due date).       */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyRenewal, ods_v2.Policy, ods.scveadmlinkfinal, ods_v2.SourceCustom */
/*              : er                                                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_terms as
select  r.PolicyCode,
        r.LYPolicyTypeGroup,
        r.LYPolicyRenewDateAdj                                          as RenewalDate,
        r.TYPolicyCode,
        r.TYPolicyRenewDateAdj,
        r.PolicyOfferNum,
        r.PolicyRetNum,
        r.TYReportingSaleDate,
        r.RenEURPremOffer,
        r.RenEURPremInvite,
        r.RenEURFee,
        r.TYEURFees,
        r.TYEURGrossPremium,
        r.Channel,
        r.RenewalMonth,
        r.LYVehicleRegKey                                               as VehicleRegKey,
        p.ClientCode,
        c.SCV_Customer_Key,
        a.AddressKey,
        case when cast(p.PolicyCancelDate as date) >  add_months(r.LYPolicyRenewDateAdj, -12)
              and cast(p.PolicyCancelDate as date) <  r.LYPolicyRenewDateAdj
             then cast(p.PolicyCancelDate as date) end                  as CancelDate
from    ${ads.ods}.PolicyRenewal r
left join ${ads.ods}.Policy p   on p.PolicyCode = r.PolicyCode
left join (
        select  upper(trim(ChillSourceCustomerReference)) as ClientCode, min(SCV_Customer_Key) as SCV_Customer_Key
        from    ${ads.link}.scveadmlinkfinal
        where   SourceSystemId = 1 and ChillSourceCustomerReference is not null
        group by upper(trim(ChillSourceCustomerReference))
        ) c     on c.ClientCode = p.ClientCode
left join (
        select  upper(trim(SourceSystemCustomerId)) as ClientCode, min(AddressKey) as AddressKey
        from    ${ads.ods}.SourceCustomer
        where   SourceSystemId = 1
        group by upper(trim(SourceSystemCustomerId))
        ) a     on a.ClientCode = p.ClientCode
where   r.PolicyOfferNum = 1
and     r.LYPolicyRenewDateAdj >= date '${ads.start_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_pairs                                                               */
/*  Description : Candidate +/-30-day renewals: each cycle x the same customer's active sales of the sa */
/*              : me product whose term starts within 30 days of the cycle's end date, with the evidenc */
/*              : e tier (1 same code, 2 warehouse lineage, 3 same vehicle / address, 4 date only). Tem */
/*              : porary and NB Not Taken Up new codes never count; a renewal sale naming another prede */
/*              : cessor matches only that predecessor.                                                 */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_terms, ods_v2.Policy, ods_v2.PolicySale, ods.scveadmlinkfina */
/*              : l, ods_v2.SourceCustomer                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_pairs as
select  t.LYPolicyTypeGroup                                             as product,
        t.SCV_Customer_Key,
        t.PolicyCode                                                    as due_code,
        t.RenewalDate                                                   as due_date,
        s.PolicyCode                                                    as cand_code,
        concat(s.PolicyCode, '|', cast(s.ReportingSaleDate as string))  as cand_key,
        s.ReportingSaleDate                                             as cand_sale_date,
        s.Channel                                                       as cand_channel,
        case when s.PolicyCode = t.PolicyCode                                                        then 1
             when s.LYPolicyCode = t.PolicyCode or s.PolicyCode = t.TYPolicyCode                     then 2
             when t.LYPolicyTypeGroup in ('Motor', 'Van') and s.VehicleRegKey is not null
                  and s.VehicleRegKey = t.VehicleRegKey                                             then 3
             when t.LYPolicyTypeGroup = 'Home' and sa.AddressKey is not null and sa.AddressKey = t.AddressKey then 3
             else 4 end                                                 as tier,
        abs(datediff(add_months(s.PolicyRenewDateAdj, -12), coalesce(t.CancelDate, t.RenewalDate))) as gap_days,
        case when t.CancelDate is null or t.CancelDate >= date_sub(t.RenewalDate, 30) then 1 else 2 end as link_rank
from    ${ads.stg}.adsv2_renewal_terms t
join    ${ads.ods}.Policy sp            on sp.PolicyTypeGroup = t.LYPolicyTypeGroup
join    ${ads.ods}.PolicySale s         on s.PolicyCode = sp.PolicyCode
join    (
        select  upper(trim(ChillSourceCustomerReference)) as ClientCode, min(SCV_Customer_Key) as SCV_Customer_Key
        from    ${ads.link}.scveadmlinkfinal
        where   SourceSystemId = 1 and ChillSourceCustomerReference is not null
        group by upper(trim(ChillSourceCustomerReference))
        ) sc    on sc.ClientCode = sp.ClientCode and sc.SCV_Customer_Key = t.SCV_Customer_Key
left join (
        select  upper(trim(SourceSystemCustomerId)) as ClientCode, min(AddressKey) as AddressKey
        from    ${ads.ods}.SourceCustomer
        where   SourceSystemId = 1
        group by upper(trim(SourceSystemCustomerId))
        ) sa    on sa.ClientCode = sp.ClientCode
where   coalesce(s.SaleCount, 0) >= 1
and     s.PolicyRenewDateAdj is not null
and     add_months(s.PolicyRenewDateAdj, -12) between date_sub(coalesce(t.CancelDate, t.RenewalDate), 30)
                                                  and date_add(coalesce(t.CancelDate, t.RenewalDate), 30)
and     (s.PolicyCode = t.PolicyCode
         or (not (s.ReportingSaleType = 'Renewals' and s.LYPolicyCode is not null and s.LYPolicyCode <> t.PolicyCode)
             and coalesce(sp.PolicyStatusDesc, '') not in ('Temporary', 'NB Not Taken Up')))

In [ ]:
%%pyspark
# stg.adsv2_renewal_match: One-to-one +/-30-day matching per customer and product (segmentation V3); renewals before rebooks, best tier, nearest start, codes break ties.
#
# Reads: stg.adsv2_renewal_pairs
import pandas as pd
pairs = spark.table(f"{stg_schema}.adsv2_renewal_pairs")

def one_to_one(pdf):
    pdf = pdf.sort_values(["link_rank", "tier", "gap_days", "due_code", "cand_key"]).reset_index(drop=True)
    due_keys = pdf["due_code"] + "|" + pdf["due_date"].astype(str)
    used_due, used_cand, keep = set(), set(), []
    for i, (d, c) in enumerate(zip(due_keys, pdf["cand_key"])):
        if d in used_due or c in used_cand:
            continue
        used_due.add(d)
        used_cand.add(c)
        keep.append(i)
    return pdf.iloc[keep]

def per_partition(batches):
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True)
    out = [one_to_one(g) for _, g in pdf.groupby(["product", "SCV_Customer_Key"], sort=False, dropna=False)]
    if out:
        res = pd.concat(out, ignore_index=True)
        for c in res.columns:                       # plain object strings: newer pandas uses Arrow-backed
            if res[c].dtype.name in ("str", "string"):  # strings, which Spark 3.5's mapInPandas rejects
                res[c] = res[c].astype(object)
        yield res

(pairs.repartition(200, "product", "SCV_Customer_Key")
      .mapInPandas(per_partition, schema=pairs.schema)
      .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
      .saveAsTable(f"{stg_schema}.adsv2_renewal_match"))


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_base                                                                */
/*  Description : One row per renewal cycle: the finance flags, the +/-30-day match, the resolved renew */
/*              : al (date, policy, channel, online / call centre) and the outcome flags (lapsed, renew */
/*              : ed inside the window, lost, final).                                                   */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_terms, stg.adsv2_renewal_match, ods_v2.RenewalPortalEvent    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_base as
select  x.*,
        case when x.RenewedDate is null or x.RenewedDate > x.RenewalDate then 1 else 0 end      as Lapsed,
        case when x.RenewedDate > x.RenewalDate
              and x.RenewedDate <= date_add(x.RenewalDate, case when x.RenewalMode = 'online' then 7 else 30 end)
             then 1 else 0 end                                                                as RenewedInWindow,
        case when x.RenewedDate is null
               or x.RenewedDate > date_add(x.RenewalDate, case when x.RenewalMode = 'online' then 7 else 30 end)
             then 1 else 0 end                                                                as Lost,
        case when date_add(x.RenewalDate, 30) < date '${ads.end_time}' then 1 else 0 end      as OutcomeFinal
from    (
        select  t.*,
                case when t.CancelDate is null or t.CancelDate >= date_sub(t.RenewalDate, 30) then 1 else 0 end as DueFlag,
                case when m.cand_code is not null then 1 else 0 end                             as OpsRetained,
                m.cand_code                                                                     as OpsNewPolicyCode,
                m.cand_sale_date                                                                as OpsNewSaleDate,
                m.tier                                                                          as OpsMatchTier,
                case when t.PolicyRetNum = 1 or m.cand_code is not null then 1 else 0 end       as Renewed,
                cast(case when t.PolicyRetNum = 1 then t.TYReportingSaleDate else m.cand_sale_date end as date) as RenewedDate,
                case when t.PolicyRetNum = 1 then t.TYPolicyCode else m.cand_code end           as RenewedPolicyCode,
                case when t.PolicyRetNum = 1 then t.Channel else m.cand_channel end             as RenewedChannel,
                case when (case when t.PolicyRetNum = 1 then t.TYPolicyCode else m.cand_code end) = t.PolicyCode
                     then 1 else 0 end                                                          as RenewedSameCode,
                case when (case when t.PolicyRetNum = 1 then t.Channel else m.cand_channel end) = 'e) Online'
                       or o.PolicyCode is not null then 'online' else 'call centre' end         as RenewalMode
        from    ${ads.stg}.adsv2_renewal_terms t
        left join ${ads.stg}.adsv2_renewal_match m
                on m.product = t.LYPolicyTypeGroup and m.due_code = t.PolicyCode and m.due_date = t.RenewalDate
        left join (
                select  distinct b.PolicyCode, b.RenewalDate
                from    ${ads.stg}.adsv2_renewal_terms b
                join    ${ads.ods}.RenewalPortalEvent e on e.PolicyCode = b.PolicyCode
                where   e.EventType = 'PaymentSuccess'
                and     e.EventDateTime > date_sub(b.RenewalDate, 60)
                and     e.EventDateTime < date_add(b.RenewalDate, 40)
                ) o     on o.PolicyCode = t.PolicyCode and o.RenewalDate = t.RenewalDate
        ) x

## Shared: renewal_scv_quotes

Re-quotes (H8, D-E1.27): quotes by the renewing customer (SCV key, ods.scveadmlinkfinal) started between due - 60 and due + 30 days of the cycle, attached to that cycle only. 04 took every quote by the customer in 5 years (313,077 of 386,673 Motor quotes more than a year from any due date). Families keep the product's quote system.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_scv_quotes                                                          */
/*  Description : Renewal cycle x the same customer's quotes started between due - 60 and due + 30 days */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base, ods.scveadmlinkfinal, ods_v2.Quote                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_scv_quotes as
select  distinct
        b.PolicyCode,
        b.RenewalDate,
        b.LYPolicyTypeGroup,
        q.SourceSystemId                                                as QuoteSourceSystemId,
        q.SourceSystemQuoteReference                                    as QuoteReference,
        q.QuoteId,
        q.QuoteDateTime
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.link}.scveadmlinkfinal l  on l.SCV_Customer_Key = b.SCV_Customer_Key and l.ChillSourceQuoteReference is not null
join    ${ads.ods}.Quote q              on q.SourceSystemId = l.SourceSystemId
                                       and q.SourceSystemQuoteReference = case when l.SourceSystemId = 2 then lower(trim(l.ChillSourceQuoteReference))
                                                                               else upper(trim(l.ChillSourceQuoteReference)) end
where   q.QuoteDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
and     q.QuoteDateTime <  cast(date_add(b.RenewalDate, 31) as timestamp)

## Shared: hfq_priced

HFQ price responses with a premium returned (04 cell 15, stg.hfq_prices); read by Home acquisition and renewal.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hfq_priced                                                                  */
/*  Description : HFQ price responses with a premium returned (Premium > 0, Outcome PremiumReturned).   */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteHomePrice                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hfq_priced as
select  QuoteCodeReference, QuoteId, DateCreated, TotalPayable
from    ${ads.ods}.QuoteHomePrice
where   Premium > 0
and     Outcome = 'PremiumReturned'

## Shared: hfq_payment

The latest HFQ payment response per payment reference (04 cell 33, stg.HFQ_Payments); read by Home acquisition and renewal.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hfq_payment                                                                 */
/*  Description : The latest HFQ payment response per payment reference, with its quote (stg.HFQ_Paymen */
/*              : ts).                                                                                  */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteHomePaymentResponse                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hfq_payment as
select  QuoteReference, QuoteId, ActionResultCode, ActionTimeCreated
from    (
        select  QuoteReference, QuoteId, ActionResultCode, cast(ActionTimeCreated as timestamp) as ActionTimeCreated,
                row_number() over (partition by Reference order by cast(ActionTimeCreated as timestamp) desc) as rn
        from    ${ads.ods}.QuoteHomePaymentResponse
        where   Reference is not null
        ) r
where   rn = 1

In [ ]:
%%pyspark
ads_run_log("10_Common", "succeeded")
